# NINAAD WAGLE | BTECH AI SEM V | I065 B2 | NLP LAB 9

**Sentiment analysis** means reading a piece of text and deciding whether the writer feels good or bad
about something. Here we teach the computer to read a movie review and say **pos** (liked the movie)
or **neg** (did not like it).

The plan is simple: pick a dataset of reviews whose answers we already know, turn each review into
numbers, train two different classifiers on most of the reviews, and then test them on reviews they
have never seen. Because we know the real answers for the test reviews, we can measure which model
did better.

# Part A : To select a dataset for sentiment analysis

## Step 1 : Loading the movie reviews dataset

We use the **NLTK movie_reviews** dataset. It has 2000 real movie reviews, 1000 positive and 1000
negative, and each one is already labelled.

In [1]:
import sys
sys.path.append("..")   # preprocessing.py lives one level up, in NLP-Lab-Codes/

import pandas as pd
from preprocessing import load_movie_reviews, preprocess, classification_scores

texts, labels = load_movie_reviews()
reviews = pd.DataFrame({"review": texts, "label": labels})

print("number of reviews :", len(reviews))
print(reviews["label"].value_counts().to_dict())
reviews.head()

number of reviews : 2000
{'neg': 1000, 'pos': 1000}


,review,label
0,"plot : two teen couples go to a church party ,...",neg
1,the happy bastard's quick movie review \ndamn ...,neg
2,it is movies like these that make a jaded movi...,neg
3,""" quest for camelot "" is warner bros . ' firs...",neg
4,synopsis : a mentally unstable man undergoing ...,neg


## Step 2 : Cleaning the text and splitting it into train and test sets

Each review goes through `preprocess()` from `preprocessing.py`. It lowercases the text, splits it
into words, removes punctuation and stopwords (*the, is, of*), and lemmatizes (*movies* becomes
*movie*). This removes words that appear in every review and tell us nothing about the feeling.

We then keep **80%** of the reviews for training and hide **20%** for testing. `stratify` keeps the
same half positive, half negative mix in both parts.

In [2]:
from sklearn.model_selection import train_test_split

reviews["clean"] = [" ".join(preprocess(t, lemmatize=True)) for t in reviews["review"]]

X_train, X_test, y_train, y_test = train_test_split(
    reviews["clean"], reviews["label"], test_size=0.2, random_state=42, stratify=reviews["label"])

print("train reviews :", len(X_train), "| test reviews :", len(X_test))
print("cleaned sample:", reviews["clean"][3][:200], "...")

train reviews : 1600 | test reviews : 400
cleaned sample: quest camelot warner bros first feature-length fully-animated attempt steal clout disney cartoon empire mouse reason worried recent challenger throne last fall promising flawed 20th century fox produc ...


## Step 3 : Turning the text into numbers with TF-IDF

A classifier can only work with numbers, so every review becomes a **TF-IDF** vector (from Lab 2).
Each word gets a score that is high when the word is used a lot in this review but is rare across
the other reviews. Words like *brilliant* or *boring* get high scores, while words found in every
review get low ones.

The vectorizer learns its vocabulary from the **training reviews only**. The test reviews are just
converted using that vocabulary, so the test stays truly unseen.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("train matrix :", X_train_vec.shape, "| test matrix :", X_test_vec.shape)

train matrix : (1600, 32371) | test matrix : (400, 32371)


# Part B : To develop classification models using two different algorithms

## Step 1 : Training a Naive Bayes classifier

**Naive Bayes** counts how often each word shows up in positive reviews and in negative reviews. For
a new review, it multiplies these chances together and picks the class that is more likely. It is
called *naive* because it treats every word as if it had nothing to do with the words around it.

**Why this algorithm?** It is the classic starting point for text classification. It is very fast,
easy to understand, and works well even with thousands of different words.

In [4]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

print("Naive Bayes trained on", X_train_vec.shape[0], "reviews")

Naive Bayes trained on 1600 reviews


## Step 2 : Training a Logistic Regression classifier

**Logistic Regression** gives every word a weight. Words that point to a positive review get a plus
weight, words that point to a negative review get a minus weight. For a new review it adds up the
weights of its words and turns the total into a probability between 0 and 1.

**Why this algorithm?** It learns differently from Naive Bayes. Instead of just counting, it adjusts
the weights to fix its own mistakes on the training reviews, so it usually does a bit better. The
weights can also be read directly, so we can see which words the model thinks are positive or negative.

In [5]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_vec, y_train)

weights = pd.Series(lr_model.coef_[0], index=vectorizer.get_feature_names_out()).sort_values()
print("most negative words :", list(weights.index[:8]))
print("most positive words :", list(weights.index[-8:]))

most negative words : ['bad', 'worst', 'plot', 'boring', 'nothing', 'movie', 'supposed', 'script']
most positive words : ['best', 'family', 'well', 'also', 'life', 'performance', 'war', 'great']


# Part C : To predict the class of the test dataset

## Step 1 : Predicting the test reviews with both models

Both models now label the 400 test reviews they have never seen. The table puts the real label next
to each model's guess for the first few reviews.

In [6]:
nb_pred = nb_model.predict(X_test_vec)
lr_pred = lr_model.predict(X_test_vec)

predictions = pd.DataFrame({"review": X_test.str[:80], "actual": y_test,
                            "naive_bayes": nb_pred, "logistic_regression": lr_pred})
predictions.head(10)

,review,actual,naive_bayes,logistic_regression
748,dr dolittle 20th century fox running time 1 ho...,neg,neg,neg
20,spawn feature good guy bad guy lot fighting bl...,neg,neg,neg
1088,keen wisdom elderly bank robber naive ambition...,pos,pos,pos
466,running time approximately 1hr 40mins reviewed...,neg,pos,pos
491,one point movie staging opera go completely wr...,neg,neg,neg
1521,one sweetest tale ever made wonderful life n't...,pos,pos,pos
1038,sometimes movie come along fall somewhat askew...,pos,pos,pos
315,synopsis al simmons top-notch assasin guilty c...,neg,neg,neg
484,way gun brimming surprise good bad one good on...,neg,neg,neg
1833,seen february 15 1998 home video borrowed chri...,pos,pos,pos


## Step 2 : Predicting new reviews written by us

To check the models on fresh text, we write two short reviews of our own. They go through the same
cleaning and the same vectorizer before being predicted.

In [7]:
new_reviews = ["The acting was brilliant and the story was beautiful, I loved every minute of it.",
               "A boring, badly written film with terrible acting. Total waste of time."]

new_vec = vectorizer.transform([" ".join(preprocess(r, lemmatize=True)) for r in new_reviews])
pd.DataFrame({"review": new_reviews, "naive_bayes": nb_model.predict(new_vec),
              "logistic_regression": lr_model.predict(new_vec)})

,review,naive_bayes,logistic_regression
0,The acting was brilliant and the story was bea...,pos,pos
1,"A boring, badly written film with terrible act...",neg,neg


# Part D : To analyze and compare the results using metrics

## Step 1 : Comparing accuracy, precision, recall and F1 score

`classification_scores()` from `preprocessing.py` gives four numbers for each model, taking **pos**
as the class we are looking for:

- **Accuracy** : out of all test reviews, how many were labelled correctly.
- **Precision** : out of the reviews the model *called* positive, how many really were positive.
- **Recall** : out of the reviews that *really were* positive, how many the model found.
- **F1 score** : one number that balances precision and recall.

Our test set is half positive and half negative, so accuracy is a fair main score here. Precision and
recall show *what kind* of mistakes each model makes.

In [8]:
results = pd.DataFrame({"Naive Bayes": classification_scores(y_test, nb_pred),
                        "Logistic Regression": classification_scores(y_test, lr_pred)}).T
results

,accuracy,precision,recall,f1
Naive Bayes,0.8125,0.8238,0.795,0.8092
Logistic Regression,0.8250,0.8066,0.855,0.8301


## Step 2 : Looking at the mistakes with a confusion matrix

A **confusion matrix** counts the right and wrong answers. Rows are the real label and columns are
the predicted label. The diagonal holds the correct answers, and the other two boxes are the mistakes.

In [9]:
from sklearn.metrics import confusion_matrix

for name, pred in [("Naive Bayes", nb_pred), ("Logistic Regression", lr_pred)]:
    cm = pd.DataFrame(confusion_matrix(y_test, pred, labels=["pos", "neg"]),
                      index=["actual pos", "actual neg"], columns=["predicted pos", "predicted neg"])
    print(name, "\n", cm, "\n")

Naive Bayes 
             predicted pos  predicted neg
actual pos            159             41
actual neg             34            166 

Logistic Regression 
             predicted pos  predicted neg
actual pos            171             29
actual neg             41            159 



## Conclusion

- Both models got over 80% of the unseen reviews right, using only word scores and no grammar rules.
- **Logistic Regression did slightly better** on accuracy and F1. It learns a weight for every word
  and fixes its own errors while training, while Naive Bayes only counts words.
- The two models make different kinds of mistakes. Naive Bayes has **higher precision**: when it says
  positive, it is usually right, but it misses more positive reviews. Logistic Regression has
  **higher recall**: it finds more of the positive reviews, but also calls a few negative ones positive.
- Both models look at words one at a time, so they can be fooled by sarcasm or by phrases like
  *not good*, where the word *not* is removed as a stopword during cleaning.